In [2]:
#remove when converting to .py file
from pathlib import Path
import importlib.util

PROJECT_ROOT = Path.cwd()
helper_path = PROJECT_ROOT / ".." /"src" / "utils.py"
spec = importlib.util.spec_from_file_location("utils", helper_path)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)

In [26]:
import numpy as np
import librosa
import torch
from transformers import pipeline, ClapProcessor, ClapModel
import os
import pprint
import json
print(np.isnan(0.0))


False


In [4]:
torch.backends.cudnn.enabled = True
if torch.cuda.is_available():
    print("GPU(s):", torch.cuda.device_count(), torch.cuda.get_device_name(0))  #you need cuda otherwise set device to cpu

GPU(s): 1 NVIDIA GeForce RTX 4060 Laptop GPU


In [5]:
audio_classifier = pipeline(task="zero-shot-audio-classification", model="laion/larger_clap_general", batch=8, device='cuda')   #SET IT HERE

Device set to use cuda


In [6]:
audio_segments_path = '../segments/testSong'

In [7]:
clap_label_json = "../json/clap_labels.json"
with open(clap_label_json, 'r') as f:
    music_labels = json.load(f)

In [ ]:
result = []
threshold = 0.1 
k = 3
for segment in os.listdir(audio_segments_path):
    audio = os.path.join(audio_segments_path, segment)
    y, sr = librosa.load(audio, sr=22050)
    features = {}
    for label in music_labels:
        classes = music_labels[label]
        predictions = audio_classifier(y, candidate_labels = classes)

        top_preds = sorted(predictions, key=lambda x: x['score'], reverse=True)
        filtered = [p for p in top_preds if p['score'] >= threshold][:k]
        features[label] = filtered
    result.append(features)

In [27]:
model = ClapModel.from_pretrained("laion/larger_clap_general")
processor = ClapProcessor.from_pretrained("laion/larger_clap_general")

In [ ]:
anchor_labels = ["happy", "sad", "whimsical", "angry"]
inputs = processor(text=anchor_labels, return_tensors="pt", padding=True, truncation=True)
emb = model.get_text_features(**inputs)
emb = torch.nn.functional.normalize(emb, dim=-1)
print(emb)
with torch.no_grad():
    text_embeds = model.get_text_features(**inputs)
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

tensor([[-0.0117, -0.0809, -0.0598,  ..., -0.0225,  0.0447,  0.0254],
        [-0.0603, -0.0418, -0.0608,  ..., -0.0495,  0.0014,  0.0116],
        [-0.0316, -0.0781, -0.0893,  ..., -0.0063,  0.0500,  0.0384],
        [-0.0419, -0.0262, -0.0942,  ...,  0.0445,  0.0491,  0.0399]],
       grad_fn=<DivBackward0>)


In [9]:
pprint.pprint(result[0]['moods'])

[{'label': 'shy', 'score': 0.6930609345436096},
 {'label': 'happy', 'score': 0.17182967066764832}]


In [10]:
utils.save_as_json("clap_results", result)
#Each segment has all the weighted mood and genres.

In [11]:
def classify_clap_emotion(features:list) -> list:
    #do a softmax
    

    return